### Q2: During a big sales event, which zone has the lowest margins?
### Goal: Understand the profit per zone
### Approach:
    - find the sale price of each item during big sales events (labeled revenue)
    - find the profit of each item during big sales events (revenue - unit cost)
    - calculate the average profit margin for each zone using two methods:
        - Method 1: calculate by taking the total profit and dividing by total revenue 
        - Method 2: calculate by getting the margin for each item, then calculating the mean of those margins


### Q3: How much profit is each store getting for products on sale vs products not on sale?
### Goal: Find how much profit is going down when sales are happening 
### Approach:
    - find the sale price of each item during their non-sale weeks
    - find the sale price of each item during their sale weeks
    - calculate the profit of both (revenue - unit cost)
    - calculate the average profit margin for each store during sale and non-sale weeks using two methods:
        - Method 1: calculate by taking the total profit and dividing by total revenue
        - Method 2: calculate by getting the margin for each item, then calculating the mean of those margins

To see how both approaches were done within a single .py file, go to big_sale_event_margins.py 


### Access files necessary for getting stats

In [80]:
import pandas as pd


files = [
    "calendar.csv",   # events
    "products.csv",   # unit cost and item id
    "prices.csv",     # og sell price and marked-down sell price
    "stores.csv"      # zones
]

# open files
cldr = pd.read_csv(files[0])
prod = pd.read_csv(files[1])
prcs = pd.read_csv(files[2])
strs = pd.read_csv(files[3])

### Clean and merge data

In [81]:
# stores 131 and 214 report their prices in cents. It needs to be converted
prcs.loc[prcs["store_id"].isin([131, 214]), "sell_price"] /= 100
prcs["sell_price"] = prcs["sell_price"].round(2)


event_weeks = cldr.loc[cldr["event_name_1"].notna(), "ad_week"].unique()

# merge cost and zone into prices
df = (
    prcs.merge(prod[["item_id", "unit_cost"]], on="item_id", how="left")
    .merge(strs[["store_id", "store_name", "ad_zone"]], on="store_id", how="left")
)

# see which items have no cost, then drop them so profit and revenue cover the same rows
missing = df.loc[df["unit_cost"].isna(), "item_id"].unique()
print(len(missing), "items missing cost:", missing)
df = df.dropna(subset=["unit_cost"]).copy()


24 items missing cost: [106818 166413 225359 257150 310435 382288 431336 440542 466206 601603
 619243 689053 689405 696374 727301 752213 795863 833884 904832 937138
 945393 952559 978273 991279]


### Get variables to calculate profits and profit margins

In [82]:

keys = ["store_id", "item_id"]
df["reg_price"] = df.groupby(keys)["sell_price"].transform("max")
df["on_sale"]   = df["sell_price"] < df["reg_price"]
df["is_event"]  = df["ad_week"].isin(event_weeks)

df["profit"] = df["sell_price"] - df["unit_cost"]
df["margin"] = df["profit"] / df["sell_price"]


### Function to format the data in a good way

In [83]:
def summarize(g):
    return pd.Series({
        "profit": g["profit"].sum(),
        "revenue": g["sell_price"].sum(),
        "profit_margin_pct (method 1)": 100 * g["profit"].sum() / g["sell_price"].sum(),
        "avg_margin_pct (method 2)": 100 * g["margin"].mean(),
    })


### Get statistics for Q2
```

ad_zone               profit       revenue             profit_margin_pct                      avg_margin_pct
                                                          (Method 1)                            (Method 2)

North Louisiana    130839.06     393435.51                 33.255529                             32.630424
East Texas         174721.88     524850.48                 33.289839                             32.758641
South Arkansas     131294.64     393891.09                 33.332727                             32.734319

Lowest-margin zone: North Louisiana
```

### Get Statistics for Q3

```
                                                    profit    revenue          profit_margin_pct         avg_margin_pct
                                                                                   (Method 1)              (Method 2)
store_id store_name                     on_sale            
104      Redbud - Commerce Street       False    127974.99  368529.49              34.725848                34.554178
                                        True      32575.00  108947.25              29.899791                28.604350

117      Hollis Creek - Mill Road       False    128097.82  368573.51              34.755026                34.566428
                                        True      32619.02  109070.08              29.906478                28.584291

123      Marlow Hill - Pecan Avenue     False    123329.94  367322.35              33.575398                33.769685
                                        True      28741.35  101675.69              28.267671                27.262402

131      Sabine Crossing - Depot Street False    136338.50  380194.86              35.860164                35.321431
                                        True      32007.14  105077.53              30.460499                28.666617

208      Bayou Ridge - Levee Boulevard  False    128557.06  369499.61              34.792205                34.584514
                                        True      32441.29  108425.49              29.920354                28.329284

214      Cypress Point - College Avenue False    128552.36  369494.85              34.791381                34.584165
                                        True      32427.19  108411.45              29.911222                28.320148

226      Ouachita Bend - Parish Road    False    126862.07  367876.98              34.484917                34.448202
                                        True      31880.53  107792.37              29.575869                28.205207

305      Pinecrest - Timber Drive       False    125990.17  369796.97              34.070093                34.161039
                                        True      29635.56  102755.51              28.840848                27.645068

311      Dorsey Springs - Oak Avenue    False    128302.51  369046.73              34.765925                34.576246
                                        True      32533.42  108715.95              29.925158                28.562782

318      Millbrook - Courthouse Square  False    134131.34  377353.89              35.545238                35.084830
                                        True      31969.49  105673.69              30.253027                28.484624



on_sale      profit     revenue      profit_margin_pct          avg_margin_pct 

False    1288136.76  3707689.24              34.742307               34.564999
True      316829.99  1066545.01              29.706200               28.272340
```